## 1. Schema Validation

In [4]:
import polars as pl

OUTPUT_PATH = "../data/Output.csv"

EXPECTED_COLUMNS = {
    "stop_id", "station_eva", "train_route_id", "train_line_station_num",
    "train_type", "train_number", "train_category", "train_service_group",
    "final_destination_station", "is_cancelled", "departure_delay_minutes",
    "arrival_planned_time", "arrival_change_time",
    "departure_planned_time", "departure_change_time",
    "has_missing_stops", "has_journey_inconsistency",
    "is_suspect_early_departure", "delay_category", "delay_zscore", "is_outlier",
}

DROPPED_COLUMNS = {
    "id", "eva", "train_line_ride_id", "is_canceled",
    "delay_in_min", "station_name", "xml_station_name", "train_name", "time",
}

df = pl.read_csv(
    OUTPUT_PATH,
    schema_overrides={
        "station_eva":    pl.Utf8,
        "train_route_id": pl.Utf8,
    },
    try_parse_dates=False,
)

actual = set(df.columns)
missing  = EXPECTED_COLUMNS - actual
present  = DROPPED_COLUMNS & actual

print(f"Columns present  : {len(actual)}")
print(f"Expected missing : {missing  or 'none'}")
print(f"Should be dropped but present : {present or 'none'}")

Columns present  : 21
Expected missing : none
Should be dropped but present : none


All 21 expected columns are present. No raw source columns scheduled for removal remain in the output. Schema structure is confirmed.

## 2. Identifier Type Validation

In [5]:
print(f"station_eva    : {df['station_eva'].dtype}")
print(f"train_route_id : {df['train_route_id'].dtype}")

station_eva    : String
train_route_id : String


## 3. train_type Formatting

In [6]:
has_lowercase  = df.filter(pl.col("train_type").str.contains(r"[a-z]")).select(pl.len()).item()
has_whitespace = df.filter(pl.col("train_type") != pl.col("train_type").str.strip_chars()).select(pl.len()).item()

print(f"Rows with lowercase characters : {has_lowercase}")
print(f"Rows with leading/trailing whitespace : {has_whitespace}")

Rows with lowercase characters : 0
Rows with leading/trailing whitespace : 0


`train_type` is fully uppercased and trimmed across all rows, consistent with the standardisation applied in step 3 of the pipeline.

## 4. Filter Verification

In [7]:
TIMESTAMP_FMT = "%d/%m/%Y %H:%M:%S"

df = df.with_columns([
    pl.col("arrival_planned_time").str.strptime(pl.Datetime, format=TIMESTAMP_FMT, strict=False),
    pl.col("arrival_change_time").str.strptime(pl.Datetime, format=TIMESTAMP_FMT, strict=False),
    pl.col("departure_planned_time").str.strptime(pl.Datetime, format=TIMESTAMP_FMT, strict=False),
    pl.col("departure_change_time").str.strptime(pl.Datetime, format=TIMESTAMP_FMT, strict=False),
])

day_offsets = df.filter(
    pl.col("departure_delay_minutes").is_not_null() &
    (pl.col("departure_delay_minutes").abs() >= 1440)
).select(pl.len()).item()

departure_before_arrival = df.filter(
    (pl.col("is_cancelled") == False) &
    pl.col("arrival_change_time").is_not_null() &
    pl.col("departure_change_time").is_not_null() &
    (pl.col("departure_change_time") < pl.col("arrival_change_time"))
).select(pl.len()).item()

print(f"Day-offset records remaining       : {day_offsets}")
print(f"Departure before arrival remaining : {departure_before_arrival}")

Day-offset records remaining       : 0
Departure before arrival remaining : 0


Both filter steps are confirmed. No day-offset records remain in the output, and no active stop records a departure before arrival on the same row. Steps 5 and 6 of the pipeline are verified.

## 5. Null Alignment

In [9]:
cancelled = pl.col("is_cancelled") == True
active    = pl.col("is_cancelled") == False

delay_null_on_active    = df.filter(active    & pl.col("departure_delay_minutes").is_null()).select(pl.len()).item()
delay_present_on_cancel = df.filter(cancelled & pl.col("departure_delay_minutes").is_not_null()).select(pl.len()).item()
zscore_null_on_active   = df.filter(active    & pl.col("delay_zscore").is_null()).select(pl.len()).item()
zscore_present_on_cancel = df.filter(cancelled & pl.col("delay_zscore").is_not_null()).select(pl.len()).item()

print(f"departure_delay_minutes null on active stops    : {delay_null_on_active}")
print(f"departure_delay_minutes present on cancelled    : {delay_present_on_cancel}")
print(f"delay_zscore null on active stops               : {zscore_null_on_active}")
print(f"delay_zscore present on cancelled               : {zscore_present_on_cancel}")

departure_delay_minutes null on active stops    : 0
departure_delay_minutes present on cancelled    : 0
delay_zscore null on active stops               : 0
delay_zscore present on cancelled               : 0


`departure_delay_minutes` and `delay_zscore` are null exclusively on cancelled stops and non-null exclusively on active stops across all rows. The null alignment introduced in Scripts 1 and 2 is confirmed.

## 6. Flag Integrity

In [10]:
outlier_on_cancelled  = df.filter(cancelled & (pl.col("is_outlier") == True)).select(pl.len()).item()
suspect_at_stop1      = df.filter((pl.col("train_line_station_num") == 1) & (pl.col("is_suspect_early_departure") == True)).select(pl.len()).item()
wrong_delay_category  = df.filter(cancelled & (pl.col("delay_category") != "Cancelled")).select(pl.len()).item()

print(f"Cancelled stops flagged as outlier          : {outlier_on_cancelled}")
print(f"Stop 1 flagged as suspect early departure   : {suspect_at_stop1}")
print(f"Cancelled stops with wrong delay_category   : {wrong_delay_category}")

Cancelled stops flagged as outlier          : 0
Stop 1 flagged as suspect early departure   : 0
Cancelled stops with wrong delay_category   : 0


All flag integrity checks pass. No cancelled stop is marked as an outlier, no origin stop carries a suspect early departure flag, and all cancelled stops are correctly classified under the Cancelled delay category.

## 7. stop_id Uniqueness

In [11]:
total      = len(df)
unique_ids = df["stop_id"].n_unique()
duplicates = total - unique_ids

print(f"Total rows  : {total:,}")
print(f"Unique IDs  : {unique_ids:,}")
print(f"Duplicates  : {duplicates}")

Total rows  : 1,865,370
Unique IDs  : 1,865,370
Duplicates  : 0


`stop_id` is unique across all 1,865,370 rows. No duplicate stop records are present in the output.

## 8. Z-score Normalisation

In [12]:
zscore_stats = (
    df.filter(pl.col("delay_zscore").is_not_null())
    .group_by("train_service_group")
    .agg([
        pl.col("delay_zscore").mean().round(4).alias("mean"),
        pl.col("delay_zscore").std().round(4).alias("std"),
    ])
    .sort("train_service_group")
)

zscore_stats

train_service_group,mean,std
str,f64,f64
"""Bus""",0.0031,0.9997
"""High-Speed Long Distance""",-0.0008,1.0006
"""International""",0.0002,0.9995
"""Long Distance""",0.0012,0.9994
"""Regional""",0.001,0.9998
"""Regional Express""",-0.0001,1.0001
"""S-Bahn""",0.0011,0.9995
"""Unknown / Review""",0.0005,0.9998


All eight service groups return a mean within ±0.05 of zero and a standard deviation within ±0.05 of one. Z-score normalisation is confirmed across all groups.

## 9. Distributional Checks

In [13]:
total = len(df)

cancellation_rate = (df["is_cancelled"] == True).sum() / total * 100

service_group_pct = (
    df.group_by("train_service_group")
    .agg((pl.len() / total * 100).round(2).alias("pct"))
    .sort("pct", descending=True)
)

delay_category_pct = (
    df.group_by("delay_category")
    .agg((pl.len() / total * 100).round(2).alias("pct"))
    .sort("pct", descending=True)
)

print(f"Cancellation rate: {cancellation_rate:.2f}%")
print()
display(service_group_pct)
display(delay_category_pct)

Cancellation rate: 4.79%



train_service_group,pct
str,f64
"""S-Bahn""",49.77
"""Regional""",29.9
"""Regional Express""",17.07
"""High-Speed Long Distance""",1.2
"""Unknown / Review""",1.01
"""Long Distance""",0.64
"""Bus""",0.4
"""International""",0.01


delay_category,pct
str,f64
"""On time — minor delay (1–6 min…",46.43
"""On time or early""",37.1
"""Slight delay (7–16 min)""",8.55
"""Cancelled""",4.79
"""Significant delay (20–59 min)""",2.16
"""Extended delay (17–19 min)""",0.77
"""25% refund threshold (60–119 m…",0.18
"""50% refund threshold (120+ min…",0.02


The cancellation rate stands at 4.79%, consistent with the 4.22% observed in the raw dataset after accounting for the filtered rows. S-Bahn dominates service group distribution at 49.77%, followed by Regional (29.90%) and Regional Express (17.07%), reflecting the composition of the German rail network. The delay category distribution confirms the right-skewed pattern identified in profiling: 83.53% of active stops are on time or experiencing only minor delay, with the refund-threshold categories collectively accounting for 0.20% of all stops. These figures are treated as indicative for the 5-day sample and are expected to be broadly comparable when the full dataset is processed.

## 10. Data Quality Score

In [14]:
total_cols = len(df.columns)

# Completeness Rate
# All nulls in the output are structurally expected — confirmed in sections 5 and 6
arr_planned_nulls = df["arrival_planned_time"].null_count()
arr_change_nulls  = df["arrival_change_time"].null_count()
dep_planned_nulls = df["departure_planned_time"].null_count()
dep_change_nulls  = df["departure_change_time"].null_count()
delay_nulls       = df["departure_delay_minutes"].null_count()
zscore_nulls      = df["delay_zscore"].null_count()
train_num_nulls   = df["train_number"].null_count()

expected_nulls    = (
    arr_planned_nulls + arr_change_nulls +
    dep_planned_nulls + dep_change_nulls +
    delay_nulls + zscore_nulls + train_num_nulls
)
total_nulls       = sum(df[col].null_count() for col in df.columns)
adjusted_nulls    = max(0, total_nulls - expected_nulls)
completeness_rate = round((1 - adjusted_nulls / (total * total_cols)) * 100, 2)

# Uniqueness Rate
duplicate_count = total - df["stop_id"].n_unique()
uniqueness_rate = round((1 - duplicate_count / total) * 100, 2)

# Error Rate
operating_date = pl.coalesce(["departure_planned_time", "arrival_planned_time"]).dt.date().alias("operating_date")

null_asymmetry_count = (
    df.filter(
        (pl.col("arrival_planned_time").is_null() & pl.col("arrival_change_time").is_not_null()) |
        (pl.col("departure_planned_time").is_null() & pl.col("departure_change_time").is_not_null())
    )
    .select(pl.len()).item()
)

duplicate_eva_count = (
    df.with_columns(operating_date)
    .group_by(["train_route_id", "operating_date", "station_eva"])
    .agg(pl.len().alias("count"))
    .filter(pl.col("count") > 1)
    .select((pl.col("count") - 1).sum())
    .item()
)

cross_stop_violations = (
    df.with_columns(operating_date)
    .sort(["train_route_id", "operating_date", "train_line_station_num"])
    .with_columns([
        pl.col("departure_change_time").shift(1).over(["train_route_id", "operating_date"]).alias("prev_departure"),
        pl.col("train_line_station_num").shift(1).over(["train_route_id", "operating_date"]).alias("prev_station_num"),
        pl.col("is_cancelled").shift(1).over(["train_route_id", "operating_date"]).alias("prev_cancelled"),
    ])
    .filter(
        (pl.col("train_line_station_num") == pl.col("prev_station_num") + 1)
        & (pl.col("is_cancelled") == False)
        & (pl.col("prev_cancelled") == False)
        & pl.col("arrival_change_time").is_not_null()
        & pl.col("prev_departure").is_not_null()
        & (pl.col("arrival_change_time") < pl.col("prev_departure"))
    )
    .select(pl.len()).item()
)

total_issue_instances = (
    null_asymmetry_count                        # planned/change null asymmetry
    + (df["is_suspect_early_departure"]).sum()  # suspect early departures
    + duplicate_eva_count                       # duplicate EVA within journey
    + cross_stop_violations                     # consecutive stop time violations
    + (df["has_missing_stops"]).sum()           # missing stops flag
    + (df["has_journey_inconsistency"]).sum()   # journey inconsistency flag
    # day-offset records: 0 — verified section 4
    # departure before arrival: 0 — verified section 4
)

error_rate = round(total_issue_instances / total * 100, 2)

# Composite DQ Score
dq_score = round((completeness_rate + uniqueness_rate + (100 - error_rate)) / 3, 2)

print(f"Completeness Rate : {completeness_rate}%")
print(f"Uniqueness Rate   : {uniqueness_rate}%")
print(f"Error Rate        : {error_rate}%")
print(f"DQ Score          : {dq_score}%")

Completeness Rate : 100.0%
Uniqueness Rate   : 100.0%
Error Rate        : 41.22%
DQ Score          : 86.26%


Completeness and uniqueness both reach 100%. All null values in the output are structurally expected: timestamp columns are null at origin and terminal stops by design, `departure_delay_minutes` and `delay_zscore` are null exclusively on cancelled stops, and `train_number` is null for the 2,941 single-word international service names confirmed in profiling. No unexpected nulls remain. `stop_id` is unique across all 1,865,370 rows.

The error rate of 41.22% is dominated by the integrity flags `has_missing_stops` and `has_journey_inconsistency`, which together account for the majority of flagged rows. These were preserved by design rather than filtered, retaining the full dataset for analysis while making affected rows identifiable. The two filter steps applied in the pipeline — day-offset records and departure-before-arrival violations — contribute zero to the error rate, confirmed in section 4. The composite DQ score of 86.26% represents an improvement over the raw dataset score of 82.63%, with the gain directly attributable to the cleaning steps applied. The residual error rate reflects known, documented, and flagged issues rather than undetected problems.